In [24]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text, _tree

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

columns = [
    "buying",
    "maint",
    "doors",
    "persons",
    "lug_boot",
    "safety",
    "class"
]

df = pd.read_csv(url, names=columns)

X = pd.get_dummies(df.drop("class", axis=1))
y = df["class"]

tree = DecisionTreeClassifier(
    max_depth=5,
    max_leaf_nodes=10,
    random_state=42
)

tree.fit(X, y)

# print(export_text(tree, feature_names=list(X.columns)))

feature_names = list(X.columns)
class_names = tree.classes_

In [25]:
print('feature names\t:', end=' ')
for a in feature_names: 
    print(a, end=', ')
print('')
print('class names\t:', end=' ')
for a in class_names: 
    print(a, end=', ')
print('')

feature names	: buying_high, buying_low, buying_med, buying_vhigh, maint_high, maint_low, maint_med, maint_vhigh, doors_2, doors_3, doors_4, doors_5more, persons_2, persons_4, persons_more, lug_boot_big, lug_boot_med, lug_boot_small, safety_high, safety_low, safety_med, 
class names	: acc, good, unacc, vgood, 


In [26]:
rules = []

def extract_rules(node, conditions):
    tree_ = tree.tree_

    # leaf node
    if tree_.feature[node] == _tree.TREE_UNDEFINED:

        class_index = tree_.value[node][0].argmax()
        predicted_class = class_names[class_index]

        # intermediate fact
        if predicted_class == "unacc":
            result_fact = ("risk", "high")

        elif predicted_class == "acc":
            result_fact = ("risk", "medium")

        else:
            result_fact = ("risk", "low")

        rules.append({
            "conditions": conditions.copy(),
            "then": result_fact
        })

        return

    feature = feature_names[tree_.feature[node]]
    threshold = tree_.threshold[node]

    # left path
    left_conditions = conditions.copy()
    left_conditions.append((feature, "<=", threshold))

    extract_rules(tree_.children_left[node], left_conditions)

    # right path
    right_conditions = conditions.copy()
    right_conditions.append((feature, ">", threshold))

    extract_rules(tree_.children_right[node], right_conditions)


extract_rules(0, [])

final_rules = [

    {
        "conditions": [("risk", "==", "high")],
        "then": ("class", "unacc")
    },

    {
        "conditions": [("risk", "==", "medium")],
        "then": ("class", "acc")
    },

    {
        "conditions": [("risk", "==", "low")],
        "then": ("class", "good")
    }

]

all_rules = rules + final_rules

for i, rule in enumerate(all_rules, 1):

    print(f"RULE {i}")

    print("IF")

    for cond in rule["conditions"]:
        print("  AND", cond)

    print("THEN", rule["then"])
    print()

RULE 1
IF
  AND ('safety_low', '<=', np.float64(0.5))
  AND ('persons_2', '<=', np.float64(0.5))
  AND ('buying_vhigh', '<=', np.float64(0.5))
  AND ('buying_high', '<=', np.float64(0.5))
  AND ('maint_low', '<=', np.float64(0.5))
THEN ('risk', 'medium')

RULE 2
IF
  AND ('safety_low', '<=', np.float64(0.5))
  AND ('persons_2', '<=', np.float64(0.5))
  AND ('buying_vhigh', '<=', np.float64(0.5))
  AND ('buying_high', '<=', np.float64(0.5))
  AND ('maint_low', '>', np.float64(0.5))
THEN ('risk', 'low')

RULE 3
IF
  AND ('safety_low', '<=', np.float64(0.5))
  AND ('persons_2', '<=', np.float64(0.5))
  AND ('buying_vhigh', '<=', np.float64(0.5))
  AND ('buying_high', '>', np.float64(0.5))
  AND ('maint_vhigh', '<=', np.float64(0.5))
THEN ('risk', 'medium')

RULE 4
IF
  AND ('safety_low', '<=', np.float64(0.5))
  AND ('persons_2', '<=', np.float64(0.5))
  AND ('buying_vhigh', '<=', np.float64(0.5))
  AND ('buying_high', '>', np.float64(0.5))
  AND ('maint_vhigh', '>', np.float64(0.5))
THEN

In [ ]:
def match_condition(condition, facts):

    feature, operator, value = condition

    if feature not in facts:
        return False

    fact_value = facts[feature]

    if operator == "<=":
        return fact_value <= value

    elif operator == ">":
        return fact_value > value

    elif operator == "==":
        return fact_value == value

    return False


def forward_chain(initial_facts):

    facts = initial_facts.copy()

    fired_rules = []

    changed = True

    while changed:

        changed = False

        for i, rule in enumerate(all_rules, 1):

            # skip if fired alr
            if i in fired_rules:
                continue

            matched = True

            for condition in rule["conditions"]:

                if not match_condition(condition, facts):
                    matched = False
                    break

            if matched:

                result_key, result_value = rule["then"]

                # new fact
                facts[result_key] = result_value

                fired_rules.append(i)

                changed = True

                print(f"\nRULE {i} FIRED")
                print("IF")

                for c in rule["conditions"]:
                    print("  AND", c)

                print("THEN", rule["then"])

    return facts, fired_rules

In [28]:
facts = {
    "safety_low": 0,
    "persons_2": 0,
    "buying_vhigh": 0,
    "buying_high": 0,
    "maint_low": 1
}

In [29]:

final_facts, fired = forward_chain(facts)
for k, v in final_facts.items():
    print(k, "=", v)

print("\nFINAL CLASSIFICATION:", final_facts.get("class"))


RULE 2 FIRED
IF
  AND ('safety_low', '<=', np.float64(0.5))
  AND ('persons_2', '<=', np.float64(0.5))
  AND ('buying_vhigh', '<=', np.float64(0.5))
  AND ('buying_high', '<=', np.float64(0.5))
  AND ('maint_low', '>', np.float64(0.5))
THEN ('risk', 'low')

RULE 13 FIRED
IF
  AND ('risk', '==', 'low')
THEN ('class', 'good')
safety_low = 0
persons_2 = 0
buying_vhigh = 0
buying_high = 0
maint_low = 1
risk = low
class = good

FINAL CLASSIFICATION: good
